# Fourier Shell Correlation (FSC) Analysis

Computes and plots FSC curves between pairs of 3-D cryoEM volumes in MRC format.

**Resolution criteria:**
- **FSC = 0.143** — gold-standard half-map criterion (Rosenthal & Henderson 2003)
- **FSC = 0.500** — classic half-bit criterion

---

In [ ]:
import sys
import os

# Add the project root to the path so the fsc package is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsc.fsc_calculator import compute_fsc_from_files
from fsc.fsc_plotter   import plot_fsc, plot_fsc_grid

%matplotlib inline
print("Imports OK")

## 1 · Configure volumes and parameters

Set `MAPS_DIR` to the folder containing your MRC files.  
All unique pairwise combinations are computed automatically.

Set `REPORT_RESOLUTIONS_A` to one or more target resolutions (in Å) to extract
interpolated FSC values (for example, 3 entries) into the summary table and CSV exports.

In [ ]:
import glob
from itertools import combinations
from pathlib import Path

MAPS_DIR   = "maps"          # <- folder containing your MRC files
THRESHOLDS = [0.143, 0.5]
REPORT_RESOLUTIONS_A = [5.0, 6.0, 6.5]  # target resolution(s) in A for FSC interpolation + CSV export
NUM_SHELLS = None             # None -> box_size // 2 (auto)
ALIGN_MAP2 = True             # run real-space rigid map fit on map2 before FSC
SPEED_PRESET = "fast"        # one of: "fast", "balanced", "accurate"
ROTATIONAL_ALIGNMENT = True   # include rotational terms in rigid fitting
SUBPIXEL_ALIGNMENT = True     # retained for API compatibility
MAX_ROTATION_DEG = 0.4        # per-axis cap (deg) on fitted z/y/x Euler rotation
ROT_COARSE_STEP_DEG = 0.1     # max rotation step size per iteration (deg)
ROT_FINE_STEP_DEG = 0.05      # min rotation step size per iteration (deg)
ROT_SEARCH_TARGET_DIM = 128   # fit downsample target size (smaller is faster)
XLIM       = (10.0, 2.0)      # resolution axis limits in A  (left, right)

# Auto-discover all MRC files and build every unique pair
mrc_files = sorted(glob.glob(f"{MAPS_DIR}/*.mrc"))
PAIRS = [
    (a, b, f"{Path(a).stem}  vs  {Path(b).stem}")
    for a, b in combinations(mrc_files, 2)
]

print(f"Found {len(mrc_files)} volume(s) -> {len(PAIRS)} unique pair(s):")
for _, _, label in PAIRS:
    print(f"  {label}")
print(f"Target resolution(s) for FSC interpolation: {REPORT_RESOLUTIONS_A}")

## 2 · Compute FSC curves

Each pair is processed independently. If `ALIGN_MAP2=True`, map 2 is first fitted to map 1
using the updated ChimeraX-style real-space rigid fitting in `fsc_calculator.py`
(adaptive translation/rotation steps), then FSC is computed from the aligned pair.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

N_WORKERS = min(8, os.cpu_count() or 8)   # too many workers can slow FFT + real-space fitting

def _compute_one(args):
    """Worker function: must be module-level so it is picklable."""
    (
        path1,
        path2,
        label,
        num_shells,
        thresholds,
        align_map2,
        rotational_alignment,
        subpixel_alignment,
        max_rotation_deg,
        rot_coarse_step_deg,
        rot_fine_step_deg,
        rot_search_target_dim,
        speed_preset,
    ) = args
    from fsc.fsc_calculator import compute_fsc_from_files
    r = compute_fsc_from_files(
        path1,
        path2,
        num_shells=num_shells,
        thresholds=thresholds,
        align_map2=align_map2,
        rotational_alignment=rotational_alignment,
        subpixel_alignment=subpixel_alignment,
        max_rotation_deg=max_rotation_deg,
        coarse_angle_step_deg=rot_coarse_step_deg,
        fine_angle_step_deg=rot_fine_step_deg,
        rotation_search_target_dim=rot_search_target_dim,
        speed_preset=speed_preset,
    )
    r["label"] = label
    return r

job_args = [
    (
        p1,
        p2,
        label,
        NUM_SHELLS,
        THRESHOLDS,
        ALIGN_MAP2,
        ROTATIONAL_ALIGNMENT,
        SUBPIXEL_ALIGNMENT,
        MAX_ROTATION_DEG,
        ROT_COARSE_STEP_DEG,
        ROT_FINE_STEP_DEG,
        ROT_SEARCH_TARGET_DIM,
        SPEED_PRESET,
    )
    for p1, p2, label in PAIRS
]

results_map = {}   # keyed by label to preserve order after parallel completion
t0 = time.time()

with ProcessPoolExecutor(max_workers=min(N_WORKERS, len(job_args))) as pool:
    futures = {pool.submit(_compute_one, arg): arg[2] for arg in job_args}
    for future in as_completed(futures):
        label = futures[future]
        r = future.result()
        results_map[label] = r
        res_strs = "  ".join(
            f"d{thr:.3f} = {r['resolutions'][thr]:.2f} A" for thr in THRESHOLDS
        )
        if r.get("alignment", {}).get("applied", False):
            shift = r["alignment"]["shift_zyx"]
            rot = r["alignment"].get("rotation_zyx_deg", (0.0, 0.0, 0.0))
            refine_steps = int(r["alignment"].get("rotation_refinement_steps", 0))
            corr_after = r["alignment"].get("overall_correlation_after", np.nan)
            corr_before = r["alignment"].get("overall_correlation_before", np.nan)
            shift_str = f"shift(z,y,x)=({shift[0]:.3f}, {shift[1]:.3f}, {shift[2]:.3f})"
            rot_str = f"rot(z,y,x)=({rot[0]:.2f}, {rot[1]:.2f}, {rot[2]:.2f}) deg"
            refine_str = f"rot_refine_steps={refine_steps}"
            corr_before_str = f"corr_before={corr_before:.4f}" if np.isfinite(corr_before) else "corr_before=nan"
            corr_str = f"corr_after={corr_after:.4f}" if np.isfinite(corr_after) else "corr_after=nan"
            print(f"  ✓ {label}  ->  {res_strs}  |  {shift_str}  |  {rot_str}  |  {refine_str}  |  {corr_before_str}  |  {corr_str}")
        else:
            print(f"  ✓ {label}  ->  {res_strs}")

# Restore original pair order
results = [results_map[label] for _, _, label in PAIRS]

print(f"\nDone - {len(results)} pair(s) in {time.time() - t0:.1f} s")

## 3 · Resolution summary table

In [ ]:
def fsc_at_resolution(result, target_resolution_A):
    """Linearly interpolate FSC at a requested resolution (A)."""
    x = np.asarray(result["resolution_A"], dtype=float)
    y = np.asarray(result["fsc"], dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    if target_resolution_A < x[0] or target_resolution_A > x[-1]:
        return np.nan
    return float(np.interp(target_resolution_A, x, y))

rows = []
for r in results:
    row = {"Pair": r["label"], "Voxel size (Å)": f"{r['voxel_size']:.3f}"}
    if r.get("alignment", {}).get("applied", False):
        shift = r["alignment"]["shift_zyx"]
        rot = r["alignment"].get("rotation_zyx_deg", (0.0, 0.0, 0.0))
        row["Shift Z (vox)"] = f"{shift[0]:.3f}"
        row["Shift Y (vox)"] = f"{shift[1]:.3f}"
        row["Shift X (vox)"] = f"{shift[2]:.3f}"
        row["Rot Z (deg)"] = f"{rot[0]:.2f}"
        row["Rot Y (deg)"] = f"{rot[1]:.2f}"
        row["Rot X (deg)"] = f"{rot[2]:.2f}"
        row["Rot Refine Steps"] = int(r["alignment"].get("rotation_refinement_steps", 0))
        row["Overall Corr (Pre alignment)"] = (
            f"{r['alignment']['overall_correlation_before']:.4f}"
            if np.isfinite(r['alignment']['overall_correlation_before'])
            else "—"
        )
        row["Overall Corr After"] = (
            f"{r['alignment']['overall_correlation_after']:.4f}"
            if np.isfinite(r['alignment']['overall_correlation_after'])
            else "—"
        )
    for target_res in REPORT_RESOLUTIONS_A:
        fsc_val = fsc_at_resolution(r, float(target_res))
        col = f"FSC @ {target_res:.2f} Å"
        row[col] = f"{fsc_val:.4f}" if np.isfinite(fsc_val) else "—"
    for thr, res in r["resolutions"].items():
        col = f"d{thr:.3f} resolution (Å)"
        row[col] = f"{res:.2f}" if np.isfinite(res) else "—"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Pair")
df

## 4 · Overlay plot — heatmap-coloured curves with compact legend

Curves are coloured by a heatmap of resolution at FSC=0.143, and the legend is
automatically compacted to stay readable when many curves are present.

In [ ]:
fig, ax = plot_fsc(
    results,
    thresholds=THRESHOLDS,
    title="FSC — all pairs",
    figsize=(9, 5),
    show_annotations=False,   # too crowded for multiple curves
    xlim=XLIM,
    color_mode="heatmap",
    heatmap_threshold=0.143,
    heatmap_cmap="viridis_r",
    legend_mode="compact",
    max_legend_entries=10,
 )
plt.show()

## 5 · Individual panel grid

In [ ]:
fig, axes = plot_fsc_grid(
    results,
    thresholds=THRESHOLDS,
    ncols=4,
    figsize_per_panel=(6, 4),
    suptitle="FSC — individual panels",
    xlim=XLIM,
)
plt.show()

## 6 · Export FSC data to CSV

Exports include FSC-at-target-resolution columns defined by `REPORT_RESOLUTIONS_A`.

In [ ]:
import os

OUT_DIR = "fsc_output"
os.makedirs(OUT_DIR, exist_ok=True)

def fsc_at_resolution(result, target_resolution_A):
    """Linearly interpolate FSC at a requested resolution (A)."""
    x = np.asarray(result["resolution_A"], dtype=float)
    y = np.asarray(result["fsc"], dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]
    y = y[mask]
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    if target_resolution_A < x[0] or target_resolution_A > x[-1]:
        return np.nan
    return float(np.interp(target_resolution_A, x, y))

# Normalize target resolution list to floats once for export
report_resolutions = [float(v) for v in REPORT_RESOLUTIONS_A]

# -- per-pair FSC curve CSVs --------------------------------------------------
for r in results:
    safe_label = r["label"].replace(" ", "_").replace("/", "-").replace("  vs  ", "_vs_")
    out_path   = os.path.join(OUT_DIR, f"fsc_{safe_label}.csv")

    # Build one row per resolution threshold as extra columns
    res_cols = {f"d{thr:.3f}_resolution_A": r["resolutions"].get(thr, np.nan)
                for thr in THRESHOLDS}

    df_out = pd.DataFrame({
        "spatial_freq_cyc_per_px": r["shell_centers"],
        "resolution_A":            r["resolution_A"],
        "fsc":                     r["fsc"],
    })
    # Append resolution estimates as constant columns so they travel with the data
    for col, val in res_cols.items():
        df_out[col] = val

    # Append user-requested FSC values at specific resolution(s)
    for target_res in report_resolutions:
        col = f"fsc_at_{target_res:.2f}A"
        df_out[col] = fsc_at_resolution(r, target_res)

    # Include rigid-alignment metadata as constant columns
    if r.get("alignment", {}).get("applied", False):
        aln = r["alignment"]
        shift = aln.get("shift_zyx", (np.nan, np.nan, np.nan))
        rot = aln.get("rotation_zyx_deg", (np.nan, np.nan, np.nan))
        df_out["alignment_method"] = aln.get("method")
        df_out["speed_preset"] = aln.get("speed_preset")
        df_out["rotation_refinement_steps"] = int(aln.get("rotation_refinement_steps", 0))
        df_out["shift_z_vox"] = float(shift[0])
        df_out["shift_y_vox"] = float(shift[1])
        df_out["shift_x_vox"] = float(shift[2])
        df_out["rot_z_deg"] = float(rot[0])
        df_out["rot_y_deg"] = float(rot[1])
        df_out["rot_x_deg"] = float(rot[2])
        df_out["overall_corr_pre_alignment"] = aln.get("overall_correlation_before", np.nan)
        df_out["overall_corr_after"] = aln.get("overall_correlation_after", np.nan)
        df_out["overall_corr_gain"] = aln.get("overall_correlation_gain", np.nan)

    df_out.to_csv(out_path, index=False, float_format="%.6f")
    print(f"Saved curve: {out_path}")

# -- summary CSV - one row per pair ------------------------------------------
summary_rows = []
for r in results:
    row = {
        "pair":          r["label"],
        "volume_1":      r["path1"],
        "volume_2":      r["path2"],
        "voxel_size_A":  r["voxel_size"],
        "n_shells":      len(r["shell_centers"]),
    }
    if r.get("alignment", {}).get("applied", False):
        shift = r["alignment"]["shift_zyx"]
        rot = r["alignment"].get("rotation_zyx_deg", (np.nan, np.nan, np.nan))
        row["alignment_method"] = r["alignment"].get("method")
        row["speed_preset"] = r["alignment"].get("speed_preset")
        row["rotation_refinement_steps"] = int(r["alignment"].get("rotation_refinement_steps", 0))
        row["shift_z_vox"] = round(float(shift[0]), 4)
        row["shift_y_vox"] = round(float(shift[1]), 4)
        row["shift_x_vox"] = round(float(shift[2]), 4)
        row["rot_z_deg"] = round(float(rot[0]), 3)
        row["rot_y_deg"] = round(float(rot[1]), 3)
        row["rot_x_deg"] = round(float(rot[2]), 3)
        row["alignment_peak_corr"] = round(float(r["alignment"]["peak_correlation"]), 6)
        row["overall_corr_pre_alignment"] = round(float(r["alignment"]["overall_correlation_before"]), 6)
        row["overall_corr_after"] = round(float(r["alignment"]["overall_correlation_after"]), 6)
        row["overall_corr_gain"] = round(float(r["alignment"]["overall_correlation_gain"]), 6)
    for target_res in report_resolutions:
        col = f"fsc_at_{target_res:.2f}A"
        fsc_val = fsc_at_resolution(r, target_res)
        row[col] = round(float(fsc_val), 6) if np.isfinite(fsc_val) else None
    for thr, res in r["resolutions"].items():
        row[f"d{thr:.3f}_resolution_A"] = round(res, 3) if np.isfinite(res) else None
    summary_rows.append(row)

summary_path = os.path.join(OUT_DIR, "fsc_summary.csv")
pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
print(f"\nSaved summary: {summary_path}")
pd.DataFrame(summary_rows).set_index("pair")

## 7 · Resolution statistics across all pairs

In [ ]:
summary_df = pd.read_csv(os.path.join(OUT_DIR, "fsc_summary.csv"))

stat_cols = [c for c in summary_df.columns if c.startswith("d") and c.endswith("_resolution_A")]

stats_rows = []
for col in stat_cols:
    vals = summary_df[col].dropna()
    thr_label = col.replace("_resolution_A", "").replace("d", "FSC = ")
    stats_rows.append({
        "Threshold":   thr_label,
        "N pairs":     len(vals),
        "Min (Å)":     vals.min(),
        "Max (Å)":     vals.max(),
        "Span (Å)":    vals.max() - vals.min(),
        "Mean (Å)":    vals.mean(),
        "Std dev (Å)": vals.std(ddof=1),
    })

stats_df = pd.DataFrame(stats_rows).set_index("Threshold")
stats_df = stats_df.round(3)
stats_df